# 🎬 Japanese Movie Transcription + Local Translation — Free/Private Edition

This Colab notebook is tuned for **Japanese movie dialogue** and produces subtitle-ready SRT files without any paid translation API.

### What this edition uses
- **Transcription:** Whisper `large-v3`, running locally on the Colab GPU.
- **Translation:** `cyberagent/CAT-Translate-7b`, an open-weight Japanese↔English translation model with an MIT license, running locally on the same GPU.
- **No Gemini/OpenAI/DeepL API key, no credit card, and no per-request translation quota.** Movie audio and subtitle text stay inside the notebook runtime and your Drive.
- The local model receives surrounding dialogue so it can handle omitted Japanese subjects and maintain scene context.
- Translation and transcription are checkpointed, so a Colab disconnect can resume without starting over when output is on Drive.
- Translation progress includes a session ETA. If the model changes marker formatting, valid lines are saved immediately and only missing cues are retried.

### Important meaning of “unlimited”
The models themselves have no API request/day or token billing limits. **Free Google Colab is not unlimited:** Google can restrict GPU availability, runtime length, and usage. For genuinely unrestricted runs, use this same notebook on your own compatible NVIDIA GPU. On Colab, process one movie at a time and keep checkpoints in Drive.

## Before running
1. Choose **Runtime → Change runtime type → T4 GPU** (or a better GPU).
2. Edit **1 — Configuration**. The default local translation model supports Japanese↔English.
3. Choose **Runtime → Run all** once. No API key and no extra Run button are needed.

The first translated run downloads roughly 15 GB of 7B model weights from Hugging Face. The model is loaded in 8-bit for maximum T4-friendly quality; if memory is tight, the notebook automatically retries in 4-bit. For a smaller/faster download with slightly lower quality, set `LOCAL_TRANSLATION_MODEL` to `cyberagent/CAT-Translate-3.3b`.


In [2]:
# @title 1 — Configuration { display-mode: "form" }
# Run this notebook with Runtime -> Run all after editing these fields.

# --- Input / output ---------------------------------------------------------
INPUT_MODE = "drive"  # @param ["drive", "upload", "urls"]
DRIVE_INPUT_FOLDER = "/content/drive/MyDrive/WhisperJAV"  # @param {type:"string"}
# For URL mode, separate multiple URLs with spaces.
DOWNLOAD_URLS_TEXT = ""  # @param {type:"string"}

OUTPUT_MODE = "drive"  # @param ["drive", "download_zip"]
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/WhisperJAV/output"  # @param {type:"string"}

# --- Language / task --------------------------------------------------------
TASK = "translate"  # @param ["translate", "transcribe"]
SOURCE_LANGUAGE = "ja"  # @param {type:"string"}
TARGET_LANGUAGE = "English"  # @param {type:"string"}
TARGET_LANGUAGE_CODE = "en"  # @param {type:"string"}

# --- Maximum-quality speech recognition -----------------------------------
MODEL_NAME = "large-v3"  # @param {type:"string"}
BEAM_SIZE = 5  # @param {type:"slider", min:1, max:10, step:1}
# Add the title, plot premise, setting, character names, or relationships.
MOVIE_CONTEXT = ""  # @param {type:"string"}
# Comma-separated names/places that Whisper often gets wrong.
HOTWORDS = ""  # @param {type:"string"}
AUDIO_STREAM_INDEX = 0  # @param {type:"integer"}

# --- Completely local, free translation -----------------------------------
# CAT-Translate-7b is best quality (~15 GB download). Use cyberagent/CAT-Translate-3.3b for smaller/faster.
LOCAL_TRANSLATION_MODEL = "cyberagent/CAT-Translate-7b"  # @param {type:"string"}
# 8bit gives better quality and fits after Whisper is unloaded. 4bit uses less GPU memory.
TRANSLATION_QUANTIZATION = "8bit"  # @param ["8bit", "4bit"]
TRANSLATION_BLOCK_SIZE = 20  # @param {type:"slider", min:5, max:50, step:5}
TRANSLATION_CONTEXT_CUES = 12  # @param {type:"slider", min:4, max:30, step:1}
# Use /content for speed. A Drive path persists the large model but loads more slowly.
HF_MODEL_CACHE = "/content/huggingface-cache"  # @param {type:"string"}

# --- Subtitle layout / files -----------------------------------------------
SOURCE_MAX_CHARS_PER_CUE = 30  # @param {type:"slider", min:20, max:50, step:1}
SOURCE_MAX_CUE_SECONDS = 7.0  # @param {type:"number"}
SOURCE_MAX_GAP_SECONDS = 0.8  # @param {type:"number"}
TARGET_MAX_CHARS_PER_LINE = 42  # @param {type:"slider", min:30, max:55, step:1}
SAVE_TXT = True  # @param {type:"boolean"}
SAVE_JSON = False  # @param {type:"boolean"}
SAVE_SOURCE_SRT = True  # @param {type:"boolean"}
FORCE_REPROCESS = False  # @param {type:"boolean"}
AUTO_DISCONNECT = False  # @param {type:"boolean"}

# --- Internal normalized values --------------------------------------------
import re

INPUT_MODE = INPUT_MODE.strip().lower()
OUTPUT_MODE = OUTPUT_MODE.strip().lower()
TASK = TASK.strip().lower()
SOURCE_LANGUAGE = SOURCE_LANGUAGE.strip().lower() or None
TARGET_LANGUAGE = TARGET_LANGUAGE.strip() or "English"
TARGET_LANGUAGE_CODE = TARGET_LANGUAGE_CODE.strip().lower() or "en"
MODEL_NAME = MODEL_NAME.strip() or "large-v3"
LOCAL_TRANSLATION_MODEL = LOCAL_TRANSLATION_MODEL.strip() or "cyberagent/CAT-Translate-7b"
TRANSLATION_QUANTIZATION = TRANSLATION_QUANTIZATION.strip().lower()
HF_MODEL_CACHE = HF_MODEL_CACHE.strip() or "/content/huggingface-cache"
DOWNLOAD_URLS = [u for u in re.split(r"\s+", DOWNLOAD_URLS_TEXT.strip()) if u]

if INPUT_MODE not in {"drive", "upload", "urls"}:
    raise ValueError("INPUT_MODE must be drive, upload, or urls")
if OUTPUT_MODE not in {"drive", "download_zip"}:
    raise ValueError("OUTPUT_MODE must be drive or download_zip")
if TASK not in {"translate", "transcribe"}:
    raise ValueError("TASK must be translate or transcribe")
if TRANSLATION_QUANTIZATION not in {"8bit", "4bit"}:
    raise ValueError("TRANSLATION_QUANTIZATION must be 8bit or 4bit")
if TASK == "translate":
    pair = {SOURCE_LANGUAGE, TARGET_LANGUAGE_CODE}
    if None in pair or pair != {"ja", "en"}:
        raise ValueError(
            "The default free local model supports Japanese↔English only. "
            "Use SOURCE_LANGUAGE='ja', TARGET_LANGUAGE='English', TARGET_LANGUAGE_CODE='en'."
        )
if SOURCE_MAX_CUE_SECONDS <= 0 or SOURCE_MAX_CHARS_PER_CUE <= 0:
    raise ValueError("Subtitle cue limits must be positive")

SESSION_INPUT_FOLDER = "/content/input"
URL_INPUT_FOLDER = "/content/url_input"
SESSION_OUTPUT_FOLDER = "/content/output"
COMPUTE_TYPE_GPU = "float16"
COMPUTE_TYPE_CPU = "int8"
VIDEO_EXTENSIONS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".ts", ".m2ts", ".mpeg", ".mpg"}
AUDIO_EXTENSIONS = {".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus", ".wma", ".webm"}
SUPPORTED_EXTENSIONS = VIDEO_EXTENSIONS | AUDIO_EXTENSIONS
PIPELINE_VERSION = "quality-v8-local-free"
TRANSLATION_PROMPT_VERSION = "cat-movie-context-v1"

print("Configuration loaded")
print(f"  input={INPUT_MODE} | output={OUTPUT_MODE} | task={TASK}")
print(f"  ASR={MODEL_NAME} | source={SOURCE_LANGUAGE or 'auto'} | beam={BEAM_SIZE}")
if TASK == "translate":
    print(
        f"  local translation={LOCAL_TRANSLATION_MODEL} ({TRANSLATION_QUANTIZATION}) "
        f"-> {TARGET_LANGUAGE} ({TARGET_LANGUAGE_CODE})"
    )
    print("  API key/card: not needed")


Configuration loaded
  input=drive | output=drive | task=translate
  ASR=large-v3 | source=ja | beam=5
  local translation=cyberagent/CAT-Translate-7b (8bit) -> English (en)
  API key/card: not needed


In [3]:
# @title 2 — Install dependencies
import shutil
import subprocess
import sys


def run_checked(command):
    subprocess.run(command, check=True)


print("Installing/updating transcription dependencies...")
base_packages = [
    "faster-whisper>=1.2.0", "requests>=2.31", "yt-dlp",
]
if TASK == "translate":
    # CAT-Translate's model card targets Transformers 4.57.x. Keep v5 out to
    # avoid breaking its chat/generation behavior.
    base_packages += [
        "transformers>=4.57.1,<5", "accelerate>=1.0", "bitsandbytes>=0.46",
        "sentencepiece", "safetensors", "huggingface-hub>=0.34",
    ]
run_checked([sys.executable, "-m", "pip", "install", "-q", "-U", *base_packages])

if INPUT_MODE == "urls":
    print("Installing URL fallback tools...")
    run_checked([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "playwright", "jsbeautifier", "nest_asyncio",
    ])
    run_checked([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"])

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    run_checked(["apt-get", "-qq", "update"])
    run_checked(["apt-get", "-qq", "install", "-y", "ffmpeg"])

print("Dependencies ready.")


Installing/updating transcription dependencies...
Dependencies ready.


In [4]:
# @title 3 — Mount Drive and create folders
import json
import os
import subprocess
import time
from datetime import timedelta
from pathlib import Path

cache_uses_drive = Path(HF_MODEL_CACHE).as_posix().startswith("/content/drive/")
if INPUT_MODE == "drive" or OUTPUT_MODE == "drive" or cache_uses_drive:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

if OUTPUT_MODE == "drive":
    OUTPUT_FOLDER = DRIVE_OUTPUT_FOLDER
else:
    OUTPUT_FOLDER = SESSION_OUTPUT_FOLDER

if INPUT_MODE == "drive":
    INPUT_FOLDER = DRIVE_INPUT_FOLDER
elif INPUT_MODE == "upload":
    INPUT_FOLDER = SESSION_INPUT_FOLDER
else:
    INPUT_FOLDER = URL_INPUT_FOLDER

Path(INPUT_FOLDER).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
Path(HF_MODEL_CACHE).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = HF_MODEL_CACHE
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print(f"Input folder:  {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
if TASK == "translate":
    print(f"Model cache:   {HF_MODEL_CACHE}")
    print("Translation runs locally — no API key or credit card.")


Mounted at /content/drive
Input folder:  /content/drive/MyDrive/WhisperJAV
Output folder: /content/drive/MyDrive/WhisperJAV/output
Model cache:   /content/huggingface-cache
Translation runs locally — no API key or credit card.


In [ ]:
# @title 4 — Upload files (only when Input = upload)
if INPUT_MODE == "upload":
    import shutil
    from google.colab import files as colab_files

    print("Select one or more audio/video files:")
    uploaded = colab_files.upload()
    for uploaded_name in uploaded:
        safe_name = Path(uploaded_name).name
        source = Path(uploaded_name)
        target = Path(INPUT_FOLDER) / safe_name
        if target.exists():
            target.unlink()
        shutil.move(str(source), str(target))
        print(f"Saved -> {target}")
else:
    print("Skipping upload step.")


Skipping upload step.


In [ ]:
# @title 5 — Download media (only when Input = URLs)
# ============================================================
# DOWNLOAD MEDIA  (only runs if INPUT_MODE == "urls")
# Strategy per host:
#   native (YouTube, Vimeo …)       -> yt-dlp direct
#   streamtape.com                  -> pure HTTP (robotlink JS parse)
#   mxdrop.to / mixdrop             -> pure HTTP (Dean-Edwards unpack)
#   voe.sx                          -> pure HTTP (page-source HLS URL)
#   kamehaus / listeamed / lulu.st /
#     turtleviplay / kame.upns.one  -> pure HTTP (DoodStream pass_md5)
#   all others                      -> Playwright intercept (improved)
# ============================================================

if INPUT_MODE != "urls":
    print("Skipping download step (INPUT_MODE != 'urls').")
else:
    if not DOWNLOAD_URLS:
        print("INPUT_MODE is 'urls' but DOWNLOAD_URLS is empty.")
    else:
        import asyncio, nest_asyncio, requests, re, base64, random, string
        nest_asyncio.apply()
        from playwright.async_api import async_playwright
        from urllib.parse import urlparse
        from jsbeautifier.unpackers import packer as jspacker

        UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"
        VIDEO_EXTS = (".mp4", ".m3u8", ".mkv", ".ts")
        SKIP_HOSTS = ("google-analytics", "googleapis", "gstatic",
                      "doubleclick", "facebook", "analytics")
        # These URL patterns are preview/promo clips – filter them in the
        # Playwright fallback so we don't stop at the 15-second teaser.
        PREVIEW_MARKERS = ("promoV", "ds_promo", "/preview/", "/promo/",
                           "_preview", "_promo", "sample.", "trailer.",
                           "view_video.php")   # DoodStream preview endpoint
        PLAY_SELECTORS = [
            '[class*="dooplay"]', '[id*="dooplay"]',   # DoodStream
            ".vjs-big-play-button",                     # Video.js
            ".plyr__control--overlaid",                 # Plyr
            '[class*="jw-icon-display"]',               # JW Player
            '[class*="play-btn"]', '[id*="play-btn"]',
            '[class*="playBtn"]',  '[id*="playBtn"]',
            "video",
        ]
        YTDLP_NATIVE = ("youtube.com", "youtu.be", "vimeo.com",
                        "dailymotion.com", "twitch.tv", "bilibili.com",
                        "twitter.com", "x.com", "tiktok.com")
        # All DoodStream-based embed hosts
        DOOD_HOSTS = ("kamehaus.net", "listeamed.net", "lulu.st",
                      "turtleviplay.xyz", "kame.upns.one")

        def is_native(url):
            h = urlparse(url).netloc.lower()
            return any(h == n or h.endswith("." + n) for n in YTDLP_NATIVE)

        def safe_slug(embed_url, idx):
            """Filesystem-safe filename slug from the embed URL path component."""
            raw = embed_url.rstrip("/").split("/")[-1].lstrip("#")
            raw = raw.split("?")[0]               # strip query string
            slug = re.sub(r'[^\w\-]', '_', raw)  # replace unsafe chars
            return slug or f"video_{idx}"

        # ── Pure HTTP: Streamtape ──────────────────────────────────────────
        def resolve_streamtape(embed_url):
            try:
                r = requests.get(embed_url, headers={"User-Agent": UA}, timeout=15)
                html = r.text
                block_m = re.search(r'robotlink.{0,600}innerHTML', html, re.DOTALL)
                if block_m:
                    ctx = html[max(0, block_m.start() - 300): block_m.end() + 300]
                    m1 = re.search(r"innerHTML\s*=\s*['\"]([^'\"]{10,})['\"]", ctx)
                    m2 = re.search(r"innerHTML\s*\+=\s*['\"]([^'\"]{4,})['\"]", ctx)
                    if m1 and m2:
                        return "https:" + m1.group(1) + m2.group(1)[2:]
                m = re.search(r'(//tapecontent\.net/[^\s"\'<>]+token=[^\s"\'<>]+)', html)
                if m:
                    return "https:" + m.group(1)
            except Exception as e:
                print(f"    streamtape resolver error: {e}")
            return None

        # ── Pure HTTP: Mxdrop ──────────────────────────────────────────────
        def resolve_mxdrop(embed_url):
            try:
                r = requests.get(embed_url,
                                 headers={"User-Agent": UA, "Referer": embed_url},
                                 timeout=15)
                html = r.text
                for packed in re.findall(
                        r'(eval\(function\(p,a,c,k,e,(?:r|d)\).*?\)\))', html, re.DOTALL):
                    if jspacker.detect(packed):
                        unpacked = jspacker.unpack(packed)
                        m = re.search(
                            r'(?:wurl|src|file)\s*[=:]\s*["\']([^"\']+\.mp4[^"\']*)["\']',
                            unpacked)
                        if m:
                            return m.group(1)
            except Exception as e:
                print(f"    mxdrop resolver error: {e}")
            return None

        # ── Pure HTTP: Voe.sx ──────────────────────────────────────────────
        def resolve_voe(embed_url):
            try:
                r = requests.get(embed_url, headers={"User-Agent": UA}, timeout=15)
                html = r.text
                m = re.search(r"['\"]hls['\"]\s*:\s*['\"]([^'\"]+\.m3u8[^'\"]*)['\"]", html)
                if m:
                    return m.group(1)
                m = re.search(r'atob\(["\']([A-Za-z0-9+/=]{20,})["\']', html)
                if m:
                    decoded = base64.b64decode(m.group(1)).decode("utf-8", errors="ignore")
                    m2 = re.search(r'(https?://[^\s"\'<>]+\.m3u8[^\s"\'<>]*)', decoded)
                    if m2:
                        return m2.group(1)
            except Exception as e:
                print(f"    voe resolver error: {e}")
            return None

        # ── Pure HTTP: DoodStream variants ─────────────────────────────────
        def resolve_doodstream(embed_url):
            """
            1. Fetch embed page → extract /pass_md5/HASH/TOKEN?expiry=N from JS.
            2. GET that endpoint (reusing session cookies) → CDN base URL.
            3. Append random suffix + query params → full playable URL.
            This works for kamehaus.net, listeamed.net, lulu.st, turtleviplay.xyz.
            """
            try:
                parts  = urlparse(embed_url)
                origin = f"{parts.scheme}://{parts.netloc}"

                sess = requests.Session()
                sess.headers.update({"User-Agent": UA, "Referer": origin})

                r    = sess.get(embed_url, timeout=20)
                html = r.text

                m = re.search(r"(/pass_md5/[^\s'\"\\)]+)", html)
                if not m:
                    return None

                pass_path = m.group(1)
                pass_url  = origin + pass_path

                r2       = sess.get(pass_url, timeout=20, headers={"Referer": embed_url})
                cdn_base = r2.text.strip()

                if not cdn_base.startswith("http"):
                    return None

                rand = ''.join(random.choices(string.ascii_lowercase + string.digits, k=10))
                qi   = pass_path.find("?")
                qs   = pass_path[qi+1:] if qi != -1 else ""

                return cdn_base + rand + (f"?{qs}" if qs else "")

            except Exception as e:
                print(f"    doodstream resolver error: {e}")
            return None

        # ── Generic Playwright interceptor (fallback) ──────────────────────
        async def intercept_video_url(embed_url, wait_ms=15000):
            found = []
            async with async_playwright() as p:
                browser = await p.chromium.launch(headless=True)
                page    = await browser.new_page()

                def on_request(request):
                    u = request.url
                    if any(ext in u for ext in VIDEO_EXTS):
                        if not any(s in u for s in SKIP_HOSTS):
                            if u not in found:
                                print(f"    caught: {u[:100]}")
                                found.append(u)

                page.on("request", on_request)
                try:
                    await page.goto(embed_url, timeout=40000)
                    await page.wait_for_timeout(3000)
                    for sel in PLAY_SELECTORS:
                        try:
                            await page.click(sel, timeout=1000)
                            print(f"    clicked: {sel}")
                            break
                        except Exception:
                            pass
                    await page.wait_for_timeout(wait_ms)
                except Exception as e:
                    print(f"    page load warning: {e}")
                await browser.close()

            if not found:
                return None
            real = [u for u in found if not any(mk in u for mk in PREVIEW_MARKERS)]
            candidates = real if real else found
            result = candidates[-1]
            print(f"    selected ({len(candidates)}/{len(found)}): {result[:100]}")
            return result

        # ── Downloaders ────────────────────────────────────────────────────
        def dl_ffmpeg(video_url, referer, out_path):
            return subprocess.run([
                "ffmpeg", "-y",
                "-user_agent", UA,
                "-headers", f"Referer: {referer}\r\nOrigin: {referer}\r\n",
                "-i", video_url, "-c", "copy", out_path,
            ]).returncode

        def dl_ytdlp(video_url, referer, out_path, is_hls=False):
            cmd = [
                "yt-dlp", "--no-check-certificate",
                "--referer", referer,
                "--add-header", f"Origin: {referer}",
                "-o", out_path, "--merge-output-format", "mp4",
            ]
            if is_hls:
                cmd += ["--hls-prefer-native"]
            cmd.append(video_url)
            return subprocess.run(cmd).returncode

        # ── Main loop ──────────────────────────────────────────────────────
        Path(INPUT_FOLDER).mkdir(parents=True, exist_ok=True)
        results = {}

        for idx, embed_url in enumerate(DOWNLOAD_URLS, start=1):
            print(f"\n{'='*60}")
            print(f"[{idx}/{len(DOWNLOAD_URLS)}] {embed_url}")

            slug     = safe_slug(embed_url, idx)
            out_path = str(Path(INPUT_FOLDER) / f"{slug}.mp4")
            host     = urlparse(embed_url).netloc.lower()
            video_url = None

            # Branch 1: yt-dlp native
            if is_native(embed_url):
                print("  yt-dlp native mode")
                rc = subprocess.run([
                    "yt-dlp", "--no-playlist", "--restrict-filenames",
                    "-f", "bestaudio/best",
                    "-o", str(Path(INPUT_FOLDER) / "%(title)s.%(ext)s"),
                    embed_url,
                ]).returncode
                results[embed_url] = "done" if rc == 0 else "failed (yt-dlp)"
                print(f"  -> {results[embed_url]}")
                continue

            # Branch 2a: Streamtape pure HTTP
            if "streamtape" in host:
                print("  streamtape HTTP resolver ...")
                video_url = resolve_streamtape(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2b: Mxdrop pure HTTP
            elif "mxdrop" in host or "mixdrop" in host:
                print("  mxdrop HTTP resolver ...")
                video_url = resolve_mxdrop(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2c: Voe.sx pure HTTP
            elif "voe.sx" in host:
                print("  voe HTTP resolver ...")
                video_url = resolve_voe(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls=True)
                    if rc != 0:
                        rc = dl_ffmpeg(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 2d: DoodStream variants pure HTTP
            elif any(d in host for d in DOOD_HOSTS):
                print("  doodstream HTTP resolver ...")
                video_url = resolve_doodstream(embed_url)
                if video_url:
                    print(f"  resolved: {video_url[:80]}")
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
                    if rc != 0:
                        rc = dl_ytdlp(video_url, embed_url, out_path)
                    results[embed_url] = "done" if rc == 0 else "failed (download)"
                    print(f"  -> {results[embed_url]}")
                    continue
                print("  HTTP resolver failed, falling back to Playwright ...")

            # Branch 3: Playwright intercept (fallback for all hosts)
            print("  Playwright intercept ...")
            video_url = asyncio.get_event_loop().run_until_complete(
                intercept_video_url(embed_url)
            )

            if not video_url:
                results[embed_url] = "failed (no URL intercepted)"
                print(f"  -> {results[embed_url]}")
                continue

            is_hls = ".m3u8" in video_url

            if "streamtape" in host or "tapecontent" in video_url:
                print("  streamtape CDN -> ffmpeg")
                rc = dl_ffmpeg(video_url, embed_url, out_path)
                if rc != 0:
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls)
            elif is_hls:
                print("  HLS -> yt-dlp")
                rc = dl_ytdlp(video_url, embed_url, out_path, is_hls=True)
                if rc != 0:
                    rc = dl_ffmpeg(video_url, embed_url, out_path)
            else:
                print("  direct mp4 -> ffmpeg")
                rc = dl_ffmpeg(video_url, embed_url, out_path)
                if rc != 0:
                    rc = dl_ytdlp(video_url, embed_url, out_path, is_hls)

            results[embed_url] = "done" if rc == 0 else "failed (download)"
            print(f"  -> {results[embed_url]}")

        print("\n\n=== DOWNLOAD SUMMARY ===")
        for url, status in results.items():
            print(f"{status}  {url}")


Skipping download step (INPUT_MODE != 'urls').


In [5]:
# @title 6 — Quality, subtitle, cache, and translation helpers
import hashlib
import json
import math
import re
import tempfile
import textwrap
import time
from datetime import timedelta
from pathlib import Path


# ---------------------------------------------------------------------------
# General / cache helpers
# ---------------------------------------------------------------------------
def atomic_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(value, handle, ensure_ascii=False, indent=2)
    temporary.replace(path)


def load_json(path, default=None):
    try:
        with open(path, "r", encoding="utf-8") as handle:
            return json.load(handle)
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        return default


def stable_digest(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def file_fingerprint(path, sample_bytes=4 * 1024 * 1024):
    """Fast content fingerprint: size plus hashes of the beginning and end."""
    path = Path(path)
    size = path.stat().st_size
    digest = hashlib.sha256()
    digest.update(str(size).encode("ascii"))
    with open(path, "rb") as handle:
        digest.update(handle.read(sample_bytes))
        if size > sample_bytes:
            handle.seek(max(0, size - sample_bytes))
            digest.update(handle.read(sample_bytes))
    return {"size": size, "sample_sha256": digest.hexdigest()}


def format_timestamp(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, milliseconds = divmod(milliseconds, 3_600_000)
    minutes, milliseconds = divmod(milliseconds, 60_000)
    secs, milliseconds = divmod(milliseconds, 1_000)
    return f"{hours:02}:{minutes:02}:{secs:02},{milliseconds:03}"


def get_duration(path):
    command = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        return 0.0
    try:
        return float(result.stdout.strip())
    except ValueError:
        return 0.0


def extract_audio(video_path, output_audio, stream_index=0):
    command = [
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(video_path), "-map", f"0:a:{int(stream_index)}",
        "-vn", "-sn", "-dn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
        str(output_audio),
    ]
    subprocess.run(command, check=True)


# ---------------------------------------------------------------------------
# Whisper output -> readable subtitle cues
# ---------------------------------------------------------------------------
_JA_CHARS = r"\u3040-\u30ff\u3400-\u9fff"
_SENTENCE_END = ("。", "！", "？", "!", "?")


def normalize_source_text(text, language):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if language in {"ja", "zh", "yue"}:
        # Whisper sometimes inserts spaces between Japanese/Chinese tokens.
        text = re.sub(fr"(?<=[{_JA_CHARS}])\s+(?=[{_JA_CHARS}])", "", text)
        text = re.sub(fr"\s+([。、「」『』！？!?、])", r"\1", text)
        text = re.sub(fr"([「『（])\s+", r"\1", text)
    return text


def whisper_segment_to_dict(segment):
    words = []
    for word in (segment.words or []):
        if word.start is None or word.end is None:
            continue
        words.append({
            "start": float(word.start),
            "end": float(word.end),
            "word": word.word,
            "probability": float(word.probability),
        })
    return {
        "start": float(segment.start),
        "end": float(segment.end),
        "text": segment.text,
        "avg_logprob": float(segment.avg_logprob),
        "compression_ratio": float(segment.compression_ratio),
        "no_speech_prob": float(segment.no_speech_prob),
        "temperature": None if segment.temperature is None else float(segment.temperature),
        "words": words,
    }


def make_readable_cues(raw_segments, language, max_chars, max_seconds, max_gap):
    """Split at word timestamps; retain all speech instead of deleting long segments."""
    cues = []

    def emit(word_group, parent):
        if not word_group:
            return
        text = normalize_source_text("".join(w["word"] for w in word_group), language)
        if not text:
            return
        probabilities = [w["probability"] for w in word_group if w.get("probability") is not None]
        cues.append({
            "start": float(word_group[0]["start"]),
            "end": float(word_group[-1]["end"]),
            "text": text,
            "word_probability": (sum(probabilities) / len(probabilities)) if probabilities else None,
            "avg_logprob": parent.get("avg_logprob"),
            "no_speech_prob": parent.get("no_speech_prob"),
        })

    for parent in raw_segments:
        words = parent.get("words") or []
        if not words:
            text = normalize_source_text(parent.get("text", ""), language)
            if text:
                cues.append({
                    "start": float(parent["start"]),
                    "end": float(parent["end"]),
                    "text": text,
                    "word_probability": None,
                    "avg_logprob": parent.get("avg_logprob"),
                    "no_speech_prob": parent.get("no_speech_prob"),
                })
            continue

        current = []
        for word in words:
            if word["end"] <= word["start"]:
                continue
            if current:
                candidate = normalize_source_text(
                    "".join(w["word"] for w in current + [word]), language
                )
                gap = float(word["start"]) - float(current[-1]["end"])
                duration = float(word["end"]) - float(current[0]["start"])
                if gap > max_gap or len(candidate) > max_chars or duration > max_seconds:
                    emit(current, parent)
                    current = []
            current.append(word)
            current_text = normalize_source_text("".join(w["word"] for w in current), language)
            current_duration = float(current[-1]["end"]) - float(current[0]["start"])
            if current_text.endswith(_SENTENCE_END) and current_duration >= 0.7:
                emit(current, parent)
                current = []
        emit(current, parent)

    cues.sort(key=lambda cue: (cue["start"], cue["end"]))
    normalized = []
    previous_end = 0.0
    for cue in cues:
        cue = dict(cue)
        cue["start"] = max(0.0, cue["start"])
        cue["end"] = max(cue["start"] + 0.08, cue["end"])
        # Remove only tiny timestamp overlap; never remove text.
        if cue["start"] < previous_end and previous_end - cue["start"] <= 0.25:
            cue["start"] = previous_end
            cue["end"] = max(cue["end"], cue["start"] + 0.08)
        cue["id"] = len(normalized) + 1
        normalized.append(cue)
        previous_end = max(previous_end, cue["end"])
    return normalized


def _balanced_cjk_wrap(text, limit):
    text = text.strip()
    if len(text) <= limit:
        return text
    midpoint = len(text) / 2
    candidates = list(range(max(1, len(text) - limit), min(limit, len(text) - 1) + 1))
    if not candidates:
        return text
    punctuation = "、。！？!?」』）"
    split_at = min(
        candidates,
        key=lambda i: (0 if text[i - 1] in punctuation else 1, abs(i - midpoint)),
    )
    return text[:split_at].rstrip() + "\n" + text[split_at:].lstrip()


def _balanced_word_wrap(text, limit):
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= limit:
        return text
    words = text.split(" ")
    if len(words) == 1:
        return _balanced_cjk_wrap(text, limit)
    candidates = []
    for i in range(1, len(words)):
        left = " ".join(words[:i])
        right = " ".join(words[i:])
        overflow = max(0, len(left) - limit) + max(0, len(right) - limit)
        candidates.append((overflow, abs(len(left) - len(right)), i))
    _, _, split_at = min(candidates)
    return " ".join(words[:split_at]) + "\n" + " ".join(words[split_at:])


def wrap_subtitle(text, language, max_chars_per_line):
    if language in {"ja", "zh", "yue", "ko"}:
        return _balanced_cjk_wrap(str(text), max_chars_per_line)
    return _balanced_word_wrap(str(text), max_chars_per_line)


def write_srt(cues, output_file, field="text", language="en", max_chars_per_line=42):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w", encoding="utf-8") as handle:
        for index, cue in enumerate(cues, 1):
            text = str(cue.get(field) or cue.get("text") or "").strip()
            text = wrap_subtitle(text, language, max_chars_per_line)
            handle.write(f"{index}\n")
            handle.write(f"{format_timestamp(cue['start'])} --> {format_timestamp(cue['end'])}\n")
            handle.write(text + "\n\n")



# ---------------------------------------------------------------------------
# Completely local Japanese <-> English translation
# ---------------------------------------------------------------------------
_LANGUAGE_NAMES = {
    "ja": "Japanese", "en": "English", "es": "Spanish", "fr": "French",
    "de": "German", "it": "Italian", "pt": "Portuguese", "zh": "Chinese",
    "yue": "Cantonese", "ko": "Korean", "ru": "Russian", "ar": "Arabic",
    "hi": "Hindi", "id": "Indonesian", "th": "Thai", "vi": "Vietnamese",
}

_local_translation_model = None
_local_translation_tokenizer = None
_local_translation_precision = None


def release_local_translation_model():
    """Release local translation VRAM before Whisper is loaded again."""
    global _local_translation_model, _local_translation_tokenizer
    global _local_translation_precision
    if _local_translation_model is not None:
        del _local_translation_model
    if _local_translation_tokenizer is not None:
        del _local_translation_tokenizer
    _local_translation_model = None
    _local_translation_tokenizer = None
    _local_translation_precision = None
    try:
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def _load_translation_model_at_precision(precision):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if not torch.cuda.is_available():
        raise RuntimeError(
            "Local 7B translation requires a CUDA GPU. In Colab select "
            "Runtime > Change runtime type > T4 GPU, then run all again."
        )

    print(f"  Loading local translator {LOCAL_TRANSLATION_MODEL} in {precision}...")
    tokenizer = AutoTokenizer.from_pretrained(
        LOCAL_TRANSLATION_MODEL,
        cache_dir=HF_MODEL_CACHE,
        use_fast=True,
    )
    if precision == "8bit":
        quantization = BitsAndBytesConfig(load_in_8bit=True)
    else:
        quantization = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_TRANSLATION_MODEL,
        cache_dir=HF_MODEL_CACHE,
        device_map={"": 0},
        quantization_config=quantization,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )
    model.eval()
    model.config.use_cache = True
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return model, tokenizer


def get_local_translation_model():
    """Load once; automatically fall back from 8-bit to 4-bit on a small GPU."""
    global _local_translation_model, _local_translation_tokenizer
    global _local_translation_precision
    if _local_translation_model is not None:
        return _local_translation_model, _local_translation_tokenizer

    requested = TRANSLATION_QUANTIZATION
    try:
        model, tokenizer = _load_translation_model_at_precision(requested)
        precision = requested
    except (RuntimeError, ValueError) as error:
        if requested != "8bit":
            raise
        error_message = str(error)
        print(f"  ⚠ 8-bit translator load failed: {error_message}")
        print("  Retrying the same free model in 4-bit...")
        del error
        release_local_translation_model()
        model, tokenizer = _load_translation_model_at_precision("4bit")
        precision = "4bit"

    _local_translation_model = model
    _local_translation_tokenizer = tokenizer
    _local_translation_precision = precision
    print(f"  Local translator ready ({precision}; no API calls)")
    return model, tokenizer


_LOCAL_TRANSLATION_SYSTEM = (
    "You are a professional film subtitle translator. Translate faithfully and naturally. "
    "Preserve meaning, tone, register, slang, profanity, humor, hesitation, names, "
    "honorific/kinship nuance, and character relationships. Do not censor, summarize, "
    "explain, or invent dialogue. Japanese often omits subjects: infer them only from "
    "nearby dialogue and supplied context; when ambiguous, avoid inventing a gender."
)
_MARKER_PATTERN = re.compile(r"\[\s*(\d{1,8})\s*\]")
_ALT_MARKER_PATTERN = re.compile(
    r"(?im)^[ \t]*(?:[-*•]\s*)?(?:id\s*[:#-]?\s*)?(\d{1,8})"
    r"(?:\s*[\]\)}.:：|\-–—]\s*|\s+)"
)


def _local_translation_prompt(cues, core_ids, completed, source_name, target_name):
    first = min(core_ids) - 1
    last = max(core_ids) - 1
    context_start = max(0, first - int(TRANSLATION_CONTEXT_CUES))
    context_end = min(len(cues), last + 1 + int(TRANSLATION_CONTEXT_CUES))

    before = cues[context_start:first]
    core = [cues[cue_id - 1] for cue_id in core_ids]
    after = cues[last + 1:context_end]
    prior = []
    for cue_id in range(max(1, min(core_ids) - 8), min(core_ids)):
        saved = completed.get(str(cue_id))
        if saved:
            prior.append(f"[{cue_id:06d}] {saved['translation']}")

    lines = []
    if before:
        lines.append("CONTEXT BEFORE (understand only; do not output):")
        lines.extend(f"[{cue['id']:06d}] {cue['text']}" for cue in before)
    lines.append("LINES TO TRANSLATE (output every marker exactly once):")
    lines.extend(f"[{cue['id']:06d}] {cue['text']}" for cue in core)
    if after:
        lines.append("CONTEXT AFTER (understand only; do not output):")
        lines.extend(f"[{cue['id']:06d}] {cue['text']}" for cue in after)

    previous_text = "\n".join(prior) if prior else "(none)"
    movie_context = MOVIE_CONTEXT.strip() or "(none provided)"
    return f"""Translate the following {source_name} movie subtitle lines into natural {target_name}.

Rules:
- Output ONLY the lines under LINES TO TRANSLATE.
- Keep each [six-digit marker] unchanged and output it exactly once.
- Put one concise subtitle translation after each marker.
- Never merge, split, renumber, omit, or add subtitle markers.
- Context lines are for subject, speaker, and relationship interpretation only.
- Prefer natural movie dialogue over a literal word-for-word gloss.
- Keep each translation short enough for at most two subtitle lines.

Movie/user context: {movie_context}
Recent translated continuity:
{previous_text}

{chr(10).join(lines)}"""


def _generate_local_translation(prompt, requested_ids):
    import torch

    model, tokenizer = get_local_translation_model()
    messages = [
        {"role": "system", "content": _LOCAL_TRANSLATION_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    )
    model_device = next(model.parameters()).device
    inputs = {key: value.to(model_device) for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[-1]
    # Subtitle cues are short. This leaves ample room while bounding a malformed
    # response that never emits EOS. Do not apply a repetition penalty: markers
    # intentionally repeat the same bracket/digit tokens and were being suppressed.
    max_new_tokens = min(3072, max(160, len(requested_ids) * 64 + 64))

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0, input_length:]
    output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    del inputs, generated, new_tokens
    return output

def _clean_local_translation(value):
    value = str(value).strip()
    value = re.sub(r"^```(?:text|txt|translation|english)?\s*", "", value, flags=re.I)
    value = re.sub(r"\s*```$", "", value)
    value = re.sub(r"^[\s:：\-–—|]+", "", value)
    value = re.sub(r"^(?:translation|english)\s*[:：]\s*", "", value, flags=re.I)
    value = re.sub(r"\s+", " ", value).strip(" `\"'*_")
    # A subtitle cue should never be an essay. Reject obvious runaway output so
    # that it is retried in a smaller request instead of poisoning the SRT.
    return value if 0 < len(value) <= 300 else ""


def _ordered_unmarked_translations(raw, requested_ids):
    """Safely recover output when line breaks survived but markers did not."""
    text = re.sub(r"^```(?:text|txt|translation|english)?\s*", "", raw.strip(), flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    candidates = []
    paragraphs = [part.strip() for part in re.split(r"\n\s*\n", text) if part.strip()]
    if len(paragraphs) == len(requested_ids):
        candidates.append(paragraphs)

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines and re.fullmatch(r"(?:translations?|english(?: translation)?s?)\s*[:：]?", lines[0], re.I):
        lines = lines[1:]
    if len(lines) == len(requested_ids):
        candidates.append(lines)

    for values in candidates:
        cleaned = []
        for value in values:
            value = re.sub(r"^(?:[-*•]|\d+[.)])\s+", "", value)
            value = _clean_local_translation(value)
            if not value:
                break
            cleaned.append(value)
        if len(cleaned) == len(requested_ids):
            return dict(zip(requested_ids, cleaned))
    return {}


def _parse_marked_translations(raw, requested_ids):
    requested = set(requested_ids)
    records = []
    for match in _MARKER_PATTERN.finditer(raw):
        records.append((match.start(), match.end(), int(match.group(1))))

    # Accept common harmless variants such as "000123: text" or "ID 123 - text".
    # Only requested IDs are accepted from this looser line-start expression.
    for match in _ALT_MARKER_PATTERN.finditer(raw):
        cue_id = int(match.group(1))
        if cue_id not in requested:
            continue
        if any(match.start() < end and match.end() > start for start, end, _ in records):
            continue
        records.append((match.start(), match.end(), cue_id))
    records.sort(key=lambda item: item[0])

    parsed = {}
    for index, (_, marker_end, cue_id) in enumerate(records):
        if cue_id not in requested:
            continue
        value_end = records[index + 1][0] if index + 1 < len(records) else len(raw)
        translation = _clean_local_translation(raw[marker_end:value_end])
        if translation:
            # Last occurrence wins if the model first echoes source and then gives
            # the actual marked translation.
            parsed[cue_id] = translation

    if not parsed and len(requested_ids) > 1:
        parsed = _ordered_unmarked_translations(raw, requested_ids)

    # For one cue, an unmarked response is unambiguous.
    if len(requested_ids) == 1 and requested_ids[0] not in parsed and raw.strip():
        fallback = _MARKER_PATTERN.sub("", raw).strip()
        fallback = re.sub(r"^(?:translation|english|\d+[.)])\s*[:：]?\s*", "", fallback, flags=re.I)
        fallback = _clean_local_translation(fallback)
        if fallback:
            parsed[requested_ids[0]] = fallback
    return parsed

def translate_cues(cues, source_name, target_name, checkpoint_path):
    """Translate locally with validation, partial-result salvage, and resume."""
    source_hash = stable_digest([{"id": cue["id"], "text": cue["text"]} for cue in cues])
    signature = stable_digest({
        "prompt": TRANSLATION_PROMPT_VERSION,
        "model": LOCAL_TRANSLATION_MODEL,
        "requested_quantization": TRANSLATION_QUANTIZATION,
        "source_hash": source_hash,
        "source": source_name,
        "target": target_name,
        "context": MOVIE_CONTEXT,
        "context_cues": int(TRANSLATION_CONTEXT_CUES),
    })
    checkpoint = {} if FORCE_REPROCESS else (load_json(checkpoint_path, {}) or {})
    if checkpoint.get("signature") != signature:
        checkpoint = {"signature": signature, "completed": {}}
    completed = checkpoint.setdefault("completed", {})
    starting_count = len(completed)
    session_start = time.time()
    generation_calls = 0
    format_retries = 0

    if starting_count:
        print(f"  Resuming local translation from checkpoint: {starting_count}/{len(cues)} cues saved")

    def save_parsed(parsed, requested):
        accepted = []
        for cue_id in requested:
            translation = parsed.get(cue_id)
            if translation and str(cue_id) not in completed:
                completed[str(cue_id)] = {"translation": translation}
                accepted.append(cue_id)
        if not accepted:
            return accepted

        checkpoint["completed"] = completed
        checkpoint["actual_quantization"] = _local_translation_precision
        checkpoint["implementation"] = "marker-salvage-v2"
        atomic_write_json(checkpoint_path, checkpoint)

        session_done = max(0, len(completed) - starting_count)
        elapsed = max(0.001, time.time() - session_start)
        if session_done:
            eta_seconds = elapsed / session_done * max(0, len(cues) - len(completed))
            eta_text = str(timedelta(seconds=int(eta_seconds)))
        else:
            eta_text = "calculating"
        if len(accepted) == 1:
            accepted_text = str(accepted[0])
        else:
            accepted_text = f"{min(accepted)}-{max(accepted)} ({len(accepted)} cues)"
        print(
            f"  Saved cues {accepted_text}; total {len(completed)}/{len(cues)} "
            f"| session ETA {eta_text}"
        )
        return accepted

    def translate_ids(core_ids):
        nonlocal generation_calls, format_retries
        requested = [cue_id for cue_id in core_ids if str(cue_id) not in completed]
        if not requested:
            return
        try:
            prompt = _local_translation_prompt(
                cues, requested, completed, source_name, target_name
            )
            generation_calls += 1
            raw = _generate_local_translation(prompt, requested)
            parsed = _parse_marked_translations(raw, requested)
            accepted = save_parsed(parsed, requested)
            missing = [cue_id for cue_id in requested if str(cue_id) not in completed]
            if not missing:
                return

            format_retries += 1
            if len(missing) >= len(requested):
                if len(requested) <= 1:
                    raise ValueError(f"local model returned no usable translation for cue {requested[0]}")
                middle = len(requested) // 2
                print(
                    f"  Format recovery: no usable mapping for {len(requested)} cues; "
                    "retrying two smaller groups"
                )
                translate_ids(requested[:middle])
                translate_ids(requested[middle:])
            else:
                # Crucially, keep every valid line and regenerate only the missing
                # tail/IDs. The previous version discarded the entire good block.
                print(
                    f"  Format recovery: kept {len(accepted)}/{len(requested)}; "
                    f"retrying only {len(missing)} missing cue(s)"
                )
                translate_ids(missing)
        except RuntimeError as error:
            error_message = str(error)
            is_oom = "out of memory" in error_message.lower()
            if len(requested) <= 1:
                raise
            del error
            if is_oom:
                try:
                    import gc
                    import torch
                    gc.collect()
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            middle = len(requested) // 2
            print(f"  Local generation failed; retrying smaller groups: {error_message}")
            translate_ids(requested[:middle])
            translate_ids(requested[middle:])

    for start in range(0, len(cues), int(TRANSLATION_BLOCK_SIZE)):
        ids = [cue["id"] for cue in cues[start:start + int(TRANSLATION_BLOCK_SIZE)]]
        translate_ids(ids)

    for cue in cues:
        cue["cleaned_text"] = cue["text"]
        cue["translated"] = completed[str(cue["id"])]["translation"]

    actual_precision = (
        _local_translation_precision
        or checkpoint.get("actual_quantization")
        or TRANSLATION_QUANTIZATION
    )
    return {
        "provider": "local Hugging Face model",
        "model": LOCAL_TRANSLATION_MODEL,
        "requested_quantization": TRANSLATION_QUANTIZATION,
        "actual_quantization": actual_precision,
        "api_key_required": False,
        "generation_calls_this_session": generation_calls,
        "format_retries_this_session": format_retries,
        "resumed_cues": starting_count,
    }


# ---------------------------------------------------------------------------
# Quality report
# ---------------------------------------------------------------------------
def quality_findings(cues, translated=False):
    findings = []
    recent = []
    for cue in cues:
        flags = []
        duration = max(0.08, cue["end"] - cue["start"])
        probability = cue.get("word_probability")
        if probability is not None and probability < 0.35:
            flags.append(f"low word confidence {probability:.2f}")
        if duration > SOURCE_MAX_CUE_SECONDS + 0.5:
            flags.append(f"long unsplittable cue {duration:.1f}s")
        normalized = re.sub(r"[\W_]+", "", cue.get("text", "").lower())
        recent.append(normalized)
        if len(recent) >= 3 and normalized and recent[-3:] == [normalized] * 3:
            flags.append("same text repeated 3 times")
        if translated:
            target = str(cue.get("translated", "")).strip()
            if not target:
                flags.append("empty translation")
            if re.search(fr"[{_JA_CHARS}]", target) and TARGET_LANGUAGE_CODE not in {"ja", "zh"}:
                flags.append("translation still contains Japanese/CJK text")
            if len(target.replace("\n", "")) > TARGET_MAX_CHARS_PER_LINE * 2:
                flags.append("translation exceeds two preferred lines")
            cps = len(target.replace("\n", "")) / duration
            if cps > 24:
                flags.append(f"fast reading speed {cps:.1f} chars/s")
        if flags:
            findings.append({
                "id": cue["id"], "start": cue["start"], "end": cue["end"],
                "flags": flags, "source": cue.get("text", ""),
                "translation": cue.get("translated", "") if translated else "",
            })
    return findings


def write_quality_report(path, findings, cue_count):
    with open(path, "w", encoding="utf-8") as handle:
        handle.write("AUTOMATED SUBTITLE QUALITY REPORT\n")
        handle.write("Flags are review hints, not automatic deletions.\n")
        handle.write(f"Cues: {cue_count}\nFlags: {len(findings)}\n\n")
        for finding in findings:
            handle.write(
                f"#{finding['id']} {format_timestamp(finding['start'])} - "
                f"{format_timestamp(finding['end'])}: {', '.join(finding['flags'])}\n"
            )
            handle.write(f"  source: {finding['source']}\n")
            if finding.get("translation"):
                handle.write(f"  target: {finding['translation']}\n")
            handle.write("\n")


In [6]:
# @title 7 — Discover input media
input_root = Path(INPUT_FOLDER).resolve()
output_root = Path(OUTPUT_FOLDER).resolve()


def is_inside(child, parent):
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False


files = []
for candidate in input_root.rglob("*"):
    if not candidate.is_file() or candidate.suffix.lower() not in SUPPORTED_EXTENSIONS:
        continue
    if ".pipeline_cache" in candidate.parts:
        continue
    if input_root != output_root and is_inside(candidate, output_root):
        continue
    if candidate.stat().st_size == 0 or candidate.name.endswith(".part"):
        continue
    files.append(candidate)
files = sorted(set(files), key=lambda item: str(item).lower())

print(f"Found {len(files)} media file(s)")
for path in files:
    print(f"  - {path.relative_to(input_root)}")
if not files:
    raise FileNotFoundError(f"No supported media files found under {INPUT_FOLDER}")


Found 1 media file(s)
  - k.mp3


In [7]:
# @title 8 — Transcribe, translate, and write outputs
import gc
import traceback

from faster_whisper import WhisperModel

success = 0
failed = 0
skipped = 0
partial = 0
failed_files = []
total_duration = 0.0
overall_start = time.time()
_model = None


def gpu_available():
    try:
        import ctranslate2
        return ctranslate2.get_cuda_device_count() > 0
    except Exception:
        result = subprocess.run(["nvidia-smi"], capture_output=True)
        return result.returncode == 0


USE_GPU = gpu_available()
DEVICE = "cuda" if USE_GPU else "cpu"
COMPUTE_TYPE = COMPUTE_TYPE_GPU if USE_GPU else COMPUTE_TYPE_CPU
print(f"Transcription device: {DEVICE} ({COMPUTE_TYPE})")
if not USE_GPU:
    print("⚠ No GPU detected. large-v3 will be very slow; enable a Colab GPU runtime.")


def release_whisper_model():
    global _model
    if _model is not None:
        del _model
        _model = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def get_model():
    global _model, COMPUTE_TYPE
    if _model is None:
        # Do not keep both large models resident when processing multiple files.
        release_local_translation_model()
        print(f"Loading {MODEL_NAME}...")
        try:
            _model = WhisperModel(MODEL_NAME, device=DEVICE, compute_type=COMPUTE_TYPE)
        except RuntimeError as error:
            if USE_GPU and COMPUTE_TYPE == "float16":
                print(f"float16 load failed ({error}); retrying with int8_float16")
                COMPUTE_TYPE = "int8_float16"
                _model = WhisperModel(MODEL_NAME, device=DEVICE, compute_type=COMPUTE_TYPE)
            else:
                raise
    return _model


asr_settings = {
    "pipeline": PIPELINE_VERSION,
    "model": MODEL_NAME,
    "source_language": SOURCE_LANGUAGE,
    "beam_size": int(BEAM_SIZE),
    "movie_context": MOVIE_CONTEXT,
    "hotwords": HOTWORDS,
    "audio_stream_index": int(AUDIO_STREAM_INDEX),
    "source_max_chars": int(SOURCE_MAX_CHARS_PER_CUE),
    "source_max_seconds": float(SOURCE_MAX_CUE_SECONDS),
    "source_max_gap": float(SOURCE_MAX_GAP_SECONDS),
}
asr_signature = stable_digest(asr_settings)
pipeline_signature = stable_digest({
    "asr": asr_settings,
    "task": TASK,
    "target_language": TARGET_LANGUAGE,
    "target_code": TARGET_LANGUAGE_CODE,
    "translation_model": LOCAL_TRANSLATION_MODEL if TASK == "translate" else None,
    "translation_quantization": TRANSLATION_QUANTIZATION if TASK == "translate" else None,
    "translation_prompt": TRANSLATION_PROMPT_VERSION,
    "translation_block": int(TRANSLATION_BLOCK_SIZE),
    "translation_context": int(TRANSLATION_CONTEXT_CUES),
    "target_max_chars_per_line": int(TARGET_MAX_CHARS_PER_LINE),
    "save_txt": bool(SAVE_TXT),
    "save_json": bool(SAVE_JSON),
    "save_source_srt": bool(SAVE_SOURCE_SRT),
})

for file_index, file_path in enumerate(files, 1):
    file_start = time.time()
    temp_audio = None
    source_written = False

    try:
        relative = file_path.relative_to(input_root)
        out_dir = output_root / relative.parent
        out_dir.mkdir(parents=True, exist_ok=True)
        stem = file_path.stem
        fingerprint = file_fingerprint(file_path)

        cache_dir = output_root / ".pipeline_cache" / relative.parent / (file_path.name + ".cache")
        cache_dir.mkdir(parents=True, exist_ok=True)
        asr_cache_path = cache_dir / "transcription.json"
        translation_cache_path = cache_dir / f"translation.{TARGET_LANGUAGE_CODE}.json"
        done_path = cache_dir / "done.json"

        done = load_json(done_path, {}) or {}
        prior_outputs = [Path(path) for path in done.get("outputs", [])]
        if (
            not FORCE_REPROCESS
            and done.get("pipeline_signature") == pipeline_signature
            and done.get("fingerprint") == fingerprint
            and prior_outputs
            and all(path.exists() for path in prior_outputs)
        ):
            skipped += 1
            print(f"[{file_index}/{len(files)}] SKIPPED (matching completed run): {file_path.name}")
            continue

        print("\n" + "=" * 78)
        print(f"[{file_index}/{len(files)}] {file_path.name}")

        cache = load_json(asr_cache_path, {}) or {}
        if (
            not FORCE_REPROCESS
            and cache.get("asr_signature") == asr_signature
            and cache.get("fingerprint") == fingerprint
            and cache.get("cues")
        ):
            print("  Reusing validated transcription cache")
            cues = cache["cues"]
            raw_segments = cache.get("raw_segments", [])
            info_dict = cache.get("info", {})
            duration = float(cache.get("duration", 0.0))
        else:
            source_audio = file_path
            if file_path.suffix.lower() in VIDEO_EXTENSIONS:
                handle = tempfile.NamedTemporaryFile(prefix="movie_audio_", suffix=".wav", delete=False)
                handle.close()
                temp_audio = Path(handle.name)
                print(f"  Extracting audio stream {AUDIO_STREAM_INDEX}...")
                extract_audio(file_path, temp_audio, AUDIO_STREAM_INDEX)
                source_audio = temp_audio

            duration = get_duration(source_audio)
            print(f"  Duration: {timedelta(seconds=int(duration))}")
            language_prompts = {
                "ja": "以下は日本語映画の会話の正確な文字起こしです。自然な句読点を使用します。",
                "en": "This is an accurate transcript of dialogue from a movie, with natural punctuation.",
                "es": "Esta es una transcripción exacta del diálogo de una película, con puntuación natural.",
                "fr": "Ceci est une transcription fidèle des dialogues d'un film, avec une ponctuation naturelle.",
                "de": "Dies ist eine genaue Abschrift von Filmdialogen mit natürlicher Zeichensetzung.",
                "ko": "다음은 자연스러운 문장 부호를 사용한 영화 대사의 정확한 전사입니다.",
                "zh": "以下是电影对白的准确转录，并使用自然的标点符号。",
            }
            prompt = language_prompts.get(SOURCE_LANGUAGE)
            if MOVIE_CONTEXT:
                prompt = (prompt + " " if prompt else "") + MOVIE_CONTEXT.strip()

            generator, info = get_model().transcribe(
                str(source_audio),
                language=SOURCE_LANGUAGE,
                task="transcribe",
                log_progress=True,
                beam_size=int(BEAM_SIZE),
                best_of=5,
                patience=1.5,
                temperature=(0.0, 0.2, 0.4, 0.6),
                condition_on_previous_text=True,
                prompt_reset_on_temperature=0.5,
                compression_ratio_threshold=2.4,
                log_prob_threshold=-1.0,
                no_speech_threshold=0.6,
                vad_filter=True,
                vad_parameters={
                    "threshold": 0.35,
                    "min_speech_duration_ms": 100,
                    "min_silence_duration_ms": 500,
                    "speech_pad_ms": 500,
                },
                word_timestamps=True,
                hallucination_silence_threshold=2.0,
                initial_prompt=prompt,
                hotwords=HOTWORDS.strip() or None,
            )

            raw_segments = []
            for segment_number, segment in enumerate(generator, 1):
                raw_segments.append(whisper_segment_to_dict(segment))
                if segment_number % 100 == 0:
                    print(f"  Decoded {segment_number} Whisper segments...")
            if not raw_segments:
                raise RuntimeError("Whisper returned no speech segments")

            detected_language = info.language
            cue_language = SOURCE_LANGUAGE or detected_language
            cues = make_readable_cues(
                raw_segments,
                cue_language,
                int(SOURCE_MAX_CHARS_PER_CUE),
                float(SOURCE_MAX_CUE_SECONDS),
                float(SOURCE_MAX_GAP_SECONDS),
            )
            if not cues:
                raise RuntimeError("No non-empty subtitle cues remained after segmentation")
            info_dict = {
                "detected_language": detected_language,
                "language_probability": float(info.language_probability),
                "duration": float(info.duration),
                "duration_after_vad": float(info.duration_after_vad),
            }
            atomic_write_json(asr_cache_path, {
                "cache_version": 2,
                "fingerprint": fingerprint,
                "asr_signature": asr_signature,
                "asr_settings": asr_settings,
                "duration": duration,
                "info": info_dict,
                "raw_segments": raw_segments,
                "cues": cues,
            })
            print(f"  Transcription cached ({len(cues)} readable cues)")

        total_duration += duration
        source_code = SOURCE_LANGUAGE or info_dict.get("detected_language") or "source"
        main_srt_path = out_dir / f"{stem}.srt"
        source_srt_path = out_dir / f"{stem}.{source_code}.srt"
        txt_path = out_dir / f"{stem}.txt"
        source_txt_path = out_dir / f"{stem}.{source_code}.txt"
        json_path = out_dir / f"{stem}.json"
        report_path = out_dir / f"{stem}.quality-report.txt"
        output_paths = []

        # Save the source immediately. Translation failure will not lose ASR work.
        if SAVE_SOURCE_SRT or TASK == "translate":
            write_srt(cues, source_srt_path, "text", source_code, max(13, SOURCE_MAX_CHARS_PER_CUE // 2))
            output_paths.append(source_srt_path)
            source_written = True
        if SAVE_TXT:
            source_txt_path.write_text("\n".join(cue["text"] for cue in cues), encoding="utf-8")
            output_paths.append(source_txt_path)

        translation_info = None
        translated = TASK == "translate" and source_code != TARGET_LANGUAGE_CODE
        if translated:
            print(f"  Translating {len(cues)} cues locally: {source_code} -> {TARGET_LANGUAGE}")
            # CAT-Translate needs most of the T4. Release Whisper first; the ASR
            # cache means it never needs to redo this file's transcription.
            release_whisper_model()
            source_language_name = _LANGUAGE_NAMES.get(source_code, source_code)
            translation_info = translate_cues(
                cues, source_language_name, TARGET_LANGUAGE, translation_cache_path
            )
            write_srt(
                cues, main_srt_path, "translated", TARGET_LANGUAGE_CODE,
                int(TARGET_MAX_CHARS_PER_LINE),
            )
            if SAVE_TXT:
                txt_path.write_text("\n".join(cue["translated"] for cue in cues), encoding="utf-8")
                output_paths.append(txt_path)
        else:
            for cue in cues:
                cue["cleaned_text"] = cue["text"]
                cue["translated"] = cue["text"]
            write_srt(
                cues, main_srt_path, "text", source_code,
                max(13, SOURCE_MAX_CHARS_PER_CUE // 2),
            )
            if SAVE_TXT and txt_path != source_txt_path:
                txt_path.write_text("\n".join(cue["text"] for cue in cues), encoding="utf-8")
                output_paths.append(txt_path)
        output_paths.append(main_srt_path)

        findings = quality_findings(cues, translated=translated)
        write_quality_report(report_path, findings, len(cues))
        output_paths.append(report_path)

        if SAVE_JSON:
            metadata = {
                "pipeline_version": PIPELINE_VERSION,
                "file": str(file_path),
                "fingerprint": fingerprint,
                "duration_seconds": duration,
                "task": TASK,
                "asr_model": MODEL_NAME,
                "translation_model": LOCAL_TRANSLATION_MODEL if translated else None,
                "source_language": source_code,
                "target_language": TARGET_LANGUAGE if translated else None,
                "recognition": info_dict,
                "translation": translation_info,
                "quality_findings": findings,
                "segments": cues,
            }
            atomic_write_json(json_path, metadata)
            output_paths.append(json_path)

        output_paths = sorted(set(Path(path) for path in output_paths), key=str)
        atomic_write_json(done_path, {
            "pipeline_signature": pipeline_signature,
            "fingerprint": fingerprint,
            "outputs": [str(path) for path in output_paths],
            "completed_at_unix": time.time(),
        })

        success += 1
        elapsed = time.time() - overall_start
        remaining = (len(files) - file_index) * (elapsed / file_index)
        print(f"  ✅ Complete in {time.time() - file_start:.1f}s | QC flags: {len(findings)}")
        print(f"  Main SRT: {main_srt_path}")
        print(f"  ETA: {timedelta(seconds=int(max(0, remaining)))}")

    except Exception as error:
        failed += 1
        if source_written:
            partial += 1
        failed_files.append({"file": str(file_path), "error": str(error)})
        print(f"  ❌ FAILED: {error}")
        traceback.print_exc(limit=3)
    finally:
        if temp_audio and temp_audio.exists():
            temp_audio.unlink()

# Free GPU memory before zipping/downloading.
release_whisper_model()
release_local_translation_model()
gc.collect()


Transcription device: cuda (float16)

[1/1] k.mp3
  Duration: 2:54:35
Loading large-v3...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

  ❌ FAILED: open() got an unexpected keyword argument 'metadata_errors'


Traceback (most recent call last):
  File "/tmp/ipykernel_774/2255838907.py", line 167, in <cell line: 0>
    generator, info = get_model().transcribe(
                      ~~~~~~~~~~~~~~~~~~~~~~^
        str(source_audio),
        ^^^^^^^^^^^^^^^^^^
    ...<22 lines>...
        hotwords=HOTWORDS.strip() or None,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/faster_whisper/transcribe.py", line 876, in transcribe
    audio = decode_audio(audio, sampling_rate=sampling_rate)
  File "/usr/local/lib/python3.13/dist-packages/faster_whisper/audio.py", line 46, in decode_audio
    with av.open(input_file, mode="r", metadata_errors="ignore") as container:
         ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: open() got an unexpected keyword argument 'metadata_errors'


0

In [ ]:
# @title 9 — Summary
elapsed = time.time() - overall_start
print("\n" + "=" * 78)
print("SUMMARY")
print("=" * 78)
print(f"Completed: {success}")
print(f"Skipped:   {skipped}")
print(f"Failed:    {failed}")
print(f"Partial:   {partial} (source transcription was preserved)")
print(f"Audio processed: {timedelta(seconds=int(total_duration))}")
print(f"Elapsed:         {timedelta(seconds=int(elapsed))}")
print(f"Outputs: {OUTPUT_FOLDER}")
if failed_files:
    print("\nFailures:")
    for item in failed_files:
        print(f"  - {item['file']}: {item['error']}")



SUMMARY
Completed: 0
Skipped:   0
Failed:    1
Partial:   1 (source transcription was preserved)
Audio processed: 1:59:54
Elapsed:         0:58:07
Outputs: /content/drive/MyDrive/WhisperJAV/output

Failures:
  - /content/drive/MyDrive/WhisperJAV/GS-2061 Memorandum play Special Selection Mother and S.mp3: local model returned no usable translation for cue 654


In [ ]:
# @title 10 — Download ZIP (only when Output = download_zip)
if OUTPUT_MODE == "download_zip":
    import zipfile
    from google.colab import files as colab_files

    zip_path = Path("/content/movie_subtitles.zip")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for output_file in Path(OUTPUT_FOLDER).rglob("*"):
            if not output_file.is_file() or ".pipeline_cache" in output_file.parts:
                continue
            archive.write(output_file, output_file.relative_to(OUTPUT_FOLDER))
    print(f"Created {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.1f} MB)")
    colab_files.download(str(zip_path))
else:
    print(f"Results are already in Google Drive: {OUTPUT_FOLDER}")


Results are already in Google Drive: /content/drive/MyDrive/WhisperJAV/output


In [ ]:
# @title 11 — Optional runtime disconnect
if AUTO_DISCONNECT and failed == 0:
    print("Disconnecting the Colab runtime...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as error:
        print(f"Could not disconnect automatically: {error}")
elif AUTO_DISCONNECT and failed:
    print("Runtime kept connected because one or more files failed.")
else:
    print("AUTO_DISCONNECT is off; runtime remains connected for review.")
